# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.37.3
Rasterio version: 1.4.3


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [5]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [6]:

EVENT_NAME = '202402_Flood_CA'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'aria_opera'      #find the name within drcs_activations OLD Directory (see link above)
PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [7]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()


# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [8]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

⚠️ S3 client initialized (limited bucket list access)
✅ Confirmed access to nasa-disasters bucket
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 6 .tif files in the S3 bucket.


['drcs_activations/202402_Flood_CA/aria_opera/OPERA_L3_DSWx-S1_provisional_20240125T020715Z_20240206T130219Z_S1A_30_v0.1_B01_WTR.tif',
 'drcs_activations/202402_Flood_CA/aria_opera/OPERA_L3_DSWx-S1_provisional_20240125T140849Z_20240206T130818Z_S1A_30_v0.1_B01_WTR.tif',
 'drcs_activations/202402_Flood_CA/aria_opera/OPERA_L3_DSWx-S1_provisional_20240206T020715Z_20240206T022545Z_S1A_30_v0.1_B01_WTR.tif',
 'drcs_activations/202402_Flood_CA/aria_opera/OPERA_L3_DSWx-S1_provisional_20240206T140849Z_20240206T134347Z_S1A_30_v0.1_B01_WTR.tif',
 'drcs_activations/202402_Flood_CA/aria_opera/water_change_map_t035_20240206T020715Z_20240125T020715Z.tif',
 'drcs_activations/202402_Flood_CA/aria_opera/water_change_map_t042_20240206T140849Z_20240125T140849Z.tif']

# For these we can see three different types of files

We will use the same rename function and place them into the same directory


## Configure bucket and paths (no need to create session manually)

In [9]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

In [13]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 256
  - Total size: 8.59 GB

📁 Cached files (first 10):
  - drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_DSWx_HLS_20240506-20240421_FloodMap.tif (1.9 MB)
  - drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS-20240506-S2B_L8_mosaic.tif (394.9 MB)
  - drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240421T133151Z_S2A_mosaic.tif (380.7 MB)
  - drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240518T132231Z_S2A_30_mosaic.tif (134.7 MB)
  - drcs_activations/202405_Flood_Brasil/aria/dswx_hls/OPERA_L3_DSWx-HLS_20240521T133151Z_S2A_30_mosaic.tif (320.0 MB)
  - drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240421_S2B_Mosaic_NoSnowIce.tif (3.1 MB)
  - drcs_activations/202405_Flood_Brasil/aria/dswx_hls/archive/OPERA_DSWx-HLS_20240506_S2B_Mosaic_NoSnowIce.tif (4.5 MB)
  - drcs_activations/202405_Flood_Brasil/aria/flood_depth/FwDET_

(256, 9225531371)

In [10]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    _
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

# Process files

In [11]:
keys

['drcs_activations/202402_Flood_CA/aria_opera/OPERA_L3_DSWx-S1_provisional_20240125T020715Z_20240206T130219Z_S1A_30_v0.1_B01_WTR.tif',
 'drcs_activations/202402_Flood_CA/aria_opera/OPERA_L3_DSWx-S1_provisional_20240125T140849Z_20240206T130818Z_S1A_30_v0.1_B01_WTR.tif',
 'drcs_activations/202402_Flood_CA/aria_opera/OPERA_L3_DSWx-S1_provisional_20240206T020715Z_20240206T022545Z_S1A_30_v0.1_B01_WTR.tif',
 'drcs_activations/202402_Flood_CA/aria_opera/OPERA_L3_DSWx-S1_provisional_20240206T140849Z_20240206T134347Z_S1A_30_v0.1_B01_WTR.tif',
 'drcs_activations/202402_Flood_CA/aria_opera/water_change_map_t035_20240206T020715Z_20240125T020715Z.tif',
 'drcs_activations/202402_Flood_CA/aria_opera/water_change_map_t042_20240206T140849Z_20240125T140849Z.tif']

In [16]:
# Define filename creator functions for different file types

def create_cog_filename_water_change_map(f, EVENT_NAME):
    """Create COG filename for water change map files with formatted timestamps."""
    f2 = Path(f).stem
    
    # Split by underscore
    parts = f2.split('_')
    
    # Find timestamp parts (YYYYMMDDTHHMMSSZ format)
    timestamps = []
    timestamp_indices = []
    
    for i, part in enumerate(parts):
        if 'T' in part and 'Z' in part and len(part) >= 16:
            # Format timestamp from YYYYMMDDTHHMMSSZ to YYYY-MM-DDTHH:MM:SSZ
            date_part = part[:8]
            time_part = part[9:15]
            formatted_ts = f"{date_part[:4]}-{date_part[4:6]}-{date_part[6:8]}T{time_part[:2]}:{time_part[2:4]}:{time_part[4:6]}Z"
            timestamps.append(formatted_ts)
            timestamp_indices.append(i)
    
    if len(timestamps) == 2:
        # Remove timestamps from original positions
        non_timestamp_parts = [parts[i] for i in range(len(parts)) if i not in timestamp_indices]
        
        # Reconstruct with formatted timestamps
        base_name = '_'.join(non_timestamp_parts)
        cog_filename = f'{EVENT_NAME}_{base_name}_{timestamps[0]}_{timestamps[1]}.tif'
    else:
        # Fallback
        cog_filename = f'{EVENT_NAME}_{f2}day.tif'
    
    return cog_filename

filter_str = ''

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]
for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_water_change_map(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202402_Flood_CA_OPERA_L3_DSWx-S1_provisional_S1A_30_v0.1_B01_WTR_2024-01-25T02:07:15Z_2024-02-06T13:02:19Z.tif
  202402_Flood_CA_OPERA_L3_DSWx-S1_provisional_S1A_30_v0.1_B01_WTR_2024-01-25T14:08:49Z_2024-02-06T13:08:18Z.tif
  202402_Flood_CA_OPERA_L3_DSWx-S1_provisional_S1A_30_v0.1_B01_WTR_2024-02-06T02:07:15Z_2024-02-06T02:25:45Z.tif
  202402_Flood_CA_OPERA_L3_DSWx-S1_provisional_S1A_30_v0.1_B01_WTR_2024-02-06T14:08:49Z_2024-02-06T13:43:47Z.tif
  202402_Flood_CA_water_change_map_t035_2024-02-06T02:07:15Z_2024-01-25T02:07:15Z.tif
  202402_Flood_CA_water_change_map_t042_2024-02-06T14:08:49Z_2024-01-25T14:08:49Z.tif


In [17]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename_water_change_map, 
                                target_dir = "Unknown", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202402_Flood_CA_OPERA_L3_DSWx-S1_provisional_S1A_30_v0.1_B01_WTR_2024-01-25T02:07:15Z_2024-02-06T13:02:19Z.tif
  202402_Flood_CA_OPERA_L3_DSWx-S1_provisional_S1A_30_v0.1_B01_WTR_2024-01-25T14:08:49Z_2024-02-06T13:08:18Z.tif
  202402_Flood_CA_OPERA_L3_DSWx-S1_provisional_S1A_30_v0.1_B01_WTR_2024-02-06T02:07:15Z_2024-02-06T02:25:45Z.tif
  202402_Flood_CA_OPERA_L3_DSWx-S1_provisional_S1A_30_v0.1_B01_WTR_2024-02-06T14:08:49Z_2024-02-06T13:43:47Z.tif
  202402_Flood_CA_water_change_map_t035_2024-02-06T02:07:15Z_2024-01-25T02:07:15Z.tif
  202402_Flood_CA_water_change_map_t042_2024-02-06T14:08:49Z_2024-01-25T14:08:49Z.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202402_Flood_CA/aria_opera
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Unknown

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202402_Flood_CA

[1/6] Processing: drcs_activations/202402_Flood_CA/aria_opera/OPERA_L3_DSWx-S1

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=255, center sample non-zero=999556/1000000
            Estimated data coverage: 59.8% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmptz74co7a_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp_uutn1ig.tif


...

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=0, center sample non-zero=1000000/1000000
            Estimated data coverage: 80.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpf8sp0lj2_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmplhojgp9k.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Unknown/202402_Flood_CA_water_change_map_t042_2024-02-06T14:08:49Z_2024-01-25T14:08:49Z.tif
   [MEMORY] Final: 739.8 MB (Change: -18.3 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202402_Flood_CA_water_change_map_t042_2024-02-06T14:08:49Z_2024-01-25T14:08:49Z.tif

✅ Batch processing complete: 6 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Unknown/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Unknown/files_converted.csv
📁 COGs saved locally to: output/202402_Flood_CA

📊 BATCH PROCESSING SUMMARY
Total files processed: 6
Successful: 6
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-09T16:28:23.753919


## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.

## Memory Usage Summary

You can check the final memory usage and cleanup

In [ ]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")